In [38]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import numpy as np
import re
import statsmodels.api as sm

In [39]:
load_dotenv()

db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}")

In [40]:
# load in dataframes
df_games = pd.read_sql("SELECT * FROM games", engine)
df_venues = pd.read_sql("SELECT * FROM venues", engine)
df_teams = pd.read_sql("SELECT * FROM teams", engine)

In [41]:
# Helper function for distance
def distance_np(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2*np.arcsin(np.sqrt(a))
    return R * c

In [42]:
# Helper function to sort rounds
def get_round_order(row):
    # Match regular rounds: 'R1', 'R23', etc.
    match = re.match(r'R(\d+)$', row['round'])
    if match:
        return int(match.group(1))
    # Finals rounds mapping (AFL convention)
    finals_order = {
        'REF': 100,  # Elimination Final (week 1)
        'RQF': 101,  # Qualifying Final (week 1)
        'RSF': 102,  # Semi Final (week 2)
        'RPF': 103,  # Preliminary Final (week 3)
        'RGF': 104,  # Grand Final (week 4)
    }
    code = row['round'][1:] 
    return finals_order.get(code, 999)  # Unknown finals get 999

In [43]:
# Merge data together
# First, split out the lat and long from a single column
df_venues[['lat', 'lon']] = df_venues['location'].str.split(',', expand=True)
df_venues['lat'] = df_venues['lat'].astype(float)
df_venues['lon'] = df_venues['lon'].astype(float)

# Merge teams with venues to get each team's home ground location
teams_home = df_teams.merge(
    df_venues[['id', 'lat', 'lon']],
    left_on='home_ground',
    right_on='id',
    suffixes=('', '_venue')
).rename(columns={'lat': 'lat_home', 'lon': 'lon_home'})
teams_home = teams_home.rename(columns={'id': 'home_team_id', 'lat': 'lat_home', 'lon': 'lon_home', 'state': 'home_state'})

# Merge games with home team location
games = df_games.merge(
    teams_home,
    left_on='home_team_id',
    right_on='home_team_id',  
    suffixes=('', '_home')
)

# Merge games with venues to get venue lat/lon
games = games.merge(
    df_venues[['id', 'lat', 'lon']],
    left_on='venue_id',
    right_on='id',
    suffixes=('', '_venue')
).rename(columns={'lat': 'lat_venue', 'lon': 'lon_venue'})

# Calculate distance
games['distance_from_home'] = distance_np(
    games['lat_home'],
    games['lon_home'],
    games['lat_venue'],
    games['lon_venue']
)

# Adding in interstate flag (calling out geelong as another state)
teams_away = df_teams[['id', 'state']].rename(columns={'id': 'away_team_id', 'state': 'away_state'})
games = games.merge(
    teams_away,
    on='away_team_id',
    how='left'
)

games['home_state'] = games.apply(lambda row: 'GEELONG' if row['home_team_id'] == 20710 else row['home_state'], axis=1)
games['away_state'] = games.apply(lambda row: 'GEELONG' if row['away_team_id'] == 20710 else row['away_state'], axis=1)

games['interstate'] = games['home_state'] != games['away_state']
games['neutral_venue'] = games['venue_id'] != games['home_ground']

print(games.head())


      id round   game_date  venue_id  home_team_id  away_team_id  season_year  \
0  13358    R3  2010-04-09        14         20709         20158         2010   
1  13359    R3  2010-04-10        14         20525         20894         2010   
2  13360    R3  2010-04-10      1323         20893             5         2010   
3  13361    R3  2010-04-10         4         19974         20434         2010   
4  13362    R3  2010-04-10         1         19882         20065         2010   

   afltables_game_id home_score_str away_score_str  ...  id_venue  lat_home  \
0              13594        10.9.69        4.17.41  ...        14  -37.8167   
1              13595      17.14.116       13.13.91  ...        14  -37.8167   
2              13596       11.15.81      16.12.108  ...        10  -34.9156   
3              13597       10.15.75       13.17.95  ...         4  -37.8199   
4              13598      16.15.111        7.14.56  ...         1  -33.8916   

   lon_home  id_venue  lat_venue  lon_

In [48]:
# Sense check information and make margin +/-, not absolute
games['margin'] = games['home_score_total'] - games['away_score_total']
avg_home_margin = games['margin'].mean()
print(f"Average home margin: {avg_home_margin:.2f}")

# Average margin by interstate flag
avg_margin_interstate = games.groupby('interstate')['margin'].mean()
print("Average home margin by interstate flag (False = intrastate, True = interstate):")
print(avg_margin_interstate)

# Average margin by neutral venue flag
avg_margin_neutral = games.groupby('neutral_venue')['margin'].mean()
print("Average home margin by neutral venue (False = true home, True = neutral):")
print(avg_margin_neutral)

# Quick model
games['interstate'] = games['interstate'].astype(int)
games['neutral_venue'] = games['neutral_venue'].astype(int)

features = ['interstate', 'neutral_venue', 'distance_from_home']
df_model = games[features + ['margin']].dropna()

X = df_model[features].astype(float)
X = sm.add_constant(X)
y = df_model['margin'].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())

Average home margin: 6.72
Average home margin by interstate flag (False = intrastate, True = interstate):
interstate
0    2.018412
1    8.532947
Name: margin, dtype: float64
Average home margin by neutral venue (False = true home, True = neutral):
neutral_venue
0    7.055150
1    5.906623
Name: margin, dtype: float64
                            OLS Regression Results                            
Dep. Variable:                 margin   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     5.776
Date:                Sat, 24 May 2025   Prob (F-statistic):           0.000618
Time:                        00:35:10   Log-Likelihood:                -16115.
No. Observations:                3115   AIC:                         3.224e+04
Df Residuals:                    3111   BIC:                         3.226e+04
Df Model:                           3            

In [50]:
# for venue_id, group in games.groupby('venue_id'):
#    avg_margin = group['margin'].mean()
#    print(f"Venue ID {venue_id}: Avg home margin = {avg_margin:.2f} (n={len(group)})")

for team_id, group in games.groupby('home_team_id'):
    avg_margin = group['margin'].mean()
    print(f"Team ID {team_id}: Avg home margin = {avg_margin:.2f} (n={len(group)})")



Team ID 5: Avg home margin = 1.80 (n=178)
Team ID 19881: Avg home margin = -3.11 (n=175)
Team ID 19882: Avg home margin = 18.44 (n=183)
Team ID 19974: Avg home margin = -4.67 (n=168)
Team ID 20065: Avg home margin = 4.27 (n=177)
Team ID 20066: Avg home margin = -9.78 (n=156)
Team ID 20157: Avg home margin = 2.84 (n=148)
Team ID 20158: Avg home margin = 10.83 (n=183)
Team ID 20433: Avg home margin = 19.03 (n=177)
Team ID 20434: Avg home margin = -2.75 (n=166)
Team ID 20525: Avg home margin = -1.09 (n=167)
Team ID 20617: Avg home margin = 12.89 (n=175)
Team ID 20709: Avg home margin = 2.14 (n=171)
Team ID 20710: Avg home margin = 30.36 (n=187)
Team ID 20802: Avg home margin = 7.67 (n=172)
Team ID 20893: Avg home margin = 9.16 (n=178)
Team ID 20894: Avg home margin = 8.83 (n=179)
Team ID 20986: Avg home margin = 7.78 (n=175)


In [51]:
print(games['margin'].describe())
print(games['margin'].head(20)) 

count    3115.000000
mean        6.715570
std        42.828308
min      -138.000000
25%       -21.000000
50%         6.000000
75%        34.000000
max       186.000000
Name: margin, dtype: float64
0     28
1     25
2    -27
3    -20
4     55
5     16
6     16
7      7
8     23
9    -40
10   -48
11    64
12    22
13   -55
14    95
15    15
16    49
17    52
18    50
19    10
Name: margin, dtype: int64
